In [1]:
!git clone https://github.com/prava241/flash-attention.git
%cd flash-attention

Cloning into 'flash-attention'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 39 (delta 12), reused 34 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 9.50 KiB | 4.75 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/flash-attention


In [ ]:
!git pull origin main
# !mkdir data
# !git checkout main

Previous HEAD position was 1affba2 timing without bounds checking
Switched to branch 'main'
Your branch is up to date with 'origin/main'.


In [61]:
import torch
import numpy as np

N = 4096
D = 256

Q = torch.randn(N, D, device="cuda")
K = torch.randn(N, D, device="cuda")
V = torch.randn(N, D, device="cuda")

Q.cpu().numpy().astype(np.float32).tofile("data/q.bin")
K.cpu().numpy().astype(np.float32).tofile("data/k.bin")
V.cpu().numpy().astype(np.float32).tofile("data/v.bin")

In [19]:
!ls data

k.bin  output.bin  q.bin  v.bin


In [94]:
# Saves output to data/output.bin, reports runtime
# TODO: Profiles for bottlenecks
!nvcc -O3 src/main.cu src/kernels.cu -o attention_test

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [95]:
!./attention_test 4096 256

baseline_attention runtime: 0 ms


In [96]:
import math

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)
start.record()

# Naive PyTorch
S = Q @ K.T
S = S / math.sqrt(D)
P = torch.softmax(S, dim=-1)
O = P @ V

#Optimized PyTorch
# O = torch.nn.functional.scaled_dot_product_attention(Q, K, V)

end.record()
torch.cuda.synchronize()
elapsed_ms = start.elapsed_time(end)

print(f"pytorch_attention runtime: {elapsed_ms:.3f} ms")

reference = O.cpu().numpy()
output = np.fromfile("data/output.bin", dtype=np.float32).reshape(N, D)

print("Max abs error:", np.max(np.abs(reference - output)))
print("Mean abs error:", np.mean(np.abs(reference - output)))
print("All close:", np.allclose(reference, output, atol=1e-5, rtol=1e-5))

pytorch_attention runtime: 8.369 ms
Max abs error: 0.1560982
Mean abs error: 0.020334747
All close: False
